# Frameflow — LTX-2.5 Video Studio on Colab

This notebook installs Wan2GP, starts the FastAPI server and HTML studio, automatically downloads/warms **LTX-2.5 Distilled 22B**, and opens a public Cloudflare URL.

It is configured for a **high-VRAM GPU** with WanGP **profile 1**. In Colab choose `Runtime → Change runtime type` and select the largest GPU available, then use `Runtime → Run all`. Profile 1 keeps the current model in VRAM; if the assigned GPU cannot hold it, use the original `wan2gp_server.ipynb` with a memory-saving profile instead.

## 1. Verify the GPU

In [ ]:
import subprocess

try:
    subprocess.run(['nvidia-smi'], check=True)
except Exception as exc:
    raise RuntimeError(
        'GPU not detected. Open Runtime → Change runtime type, select GPU, save, and rerun.'
    ) from exc

try:
    import torch
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'GPU: {gpu_name} · {vram_gb:.1f} GB VRAM')
    if vram_gb < 32:
        print('⚠ Profile 1 and Max quality are intended for high-VRAM GPUs. This GPU may run out of memory.')
except ImportError:
    print('PyTorch will be installed in step 4.')

## 2. Download or update Wan2GP

In [ ]:
from pathlib import Path
import subprocess

WAN2GP_ROOT = Path('/content/wan2gp').resolve()
REPOSITORY = 'https://github.com/hoangthvn2201/wan2gp-optimized.git'
BRANCH = 'main'

if (WAN2GP_ROOT / '.git').exists():
    subprocess.run(['git', '-C', str(WAN2GP_ROOT), 'fetch', 'origin'], check=True)
else:
    subprocess.run(['git', 'clone', REPOSITORY, str(WAN2GP_ROOT)], check=True)
subprocess.run(['git', '-C', str(WAN2GP_ROOT), 'checkout', BRANCH], check=True)
subprocess.run(['git', '-C', str(WAN2GP_ROOT), 'pull', '--ff-only', 'origin', BRANCH], check=True)

required = [
    WAN2GP_ROOT / 'wan2gp_server' / '__main__.py',
    WAN2GP_ROOT / 'wan2gp_server' / 'static' / 'index.html',
    WAN2GP_ROOT / 'defaults' / 'ltx2_25_22B_distilled.json',
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise RuntimeError(f'The selected branch is missing Frameflow/LTX-2.5 files: {missing}')
print(f'Repository ready at {WAN2GP_ROOT}')

## 3. Install system libraries

In [ ]:
import os, subprocess

install_env = os.environ.copy()
install_env['DEBIAN_FRONTEND'] = 'noninteractive'
subprocess.run(['sudo', 'apt-get', 'update', '-qq'], check=True, env=install_env)
subprocess.run([
    'sudo', 'apt-get', 'install', '-y', '--no-install-recommends',
    'ffmpeg', 'libglib2.0-0', 'libgl1', 'libportaudio2'
], check=True, env=install_env)

## 4. Install Python dependencies

In [ ]:
import os, subprocess, sys

install_env = os.environ.copy()
install_env.setdefault('DEBIAN_FRONTEND', 'noninteractive')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip', 'setuptools', 'wheel'], check=True, env=install_env)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--force-reinstall', '--no-deps',
    'torch==2.8.0', 'torchvision==0.23.0', 'torchaudio==2.8.0',
    '--index-url', 'https://download.pytorch.org/whl/cu128'
], check=True, env=install_env)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', 'xformers==0.0.32.post2',
    '--index-url', 'https://download.pytorch.org/whl/cu128'
], check=True, env=install_env)
subprocess.run([
    sys.executable, '-m', 'pip', 'install',
    '-r', str(WAN2GP_ROOT / 'requirements.txt'),
    '-r', str(WAN2GP_ROOT / 'wan2gp_server' / 'requirements.txt')
], check=True, env=install_env)
print('Python environment ready.')

## 5. Apply the Colab headless compatibility setting

In [ ]:
target = WAN2GP_ROOT / 'preprocessing/matanyone/tools/interact_tools.py'
needle = "matplotlib.use('TkAgg')"
replacement = "matplotlib.use('Agg')"
if target.exists():
    source = target.read_text()
    if needle in source:
        target.write_text(source.replace(needle, replacement, 1))
        print('Enabled the headless matplotlib backend.')
    else:
        print('Headless backend already configured or no patch is needed.')

## 6. Configure the high-VRAM LTX-2.5 server

All three generation defaults point at the same LTX-2.5 Distilled checkpoint. Profile 1 keeps the active model in VRAM. The UI exposes Draft, High, and Max canvases up to approximately 1080p and clips up to 10 seconds.

In [ ]:
import secrets

PORT = 8000
OUTPUT_DIR = Path('/content/frameflow_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SERVER_ENV = {
    'WAN2GP_ROOT': str(WAN2GP_ROOT),
    'WAN2GP_SERVER_PORT': str(PORT),
    'WAN2GP_SERVER_OUTPUT_DIR': str(OUTPUT_DIR),
    'WAN2GP_CLI_ARGS': '--profile 1 --attention sdpa',
    'WAN2GP_SERVER_T2I_MODEL': 'ltx25-distilled-image',
    'WAN2GP_SERVER_T2V_MODEL': 'ltx25-distilled',
    'WAN2GP_SERVER_I2V_MODEL': 'ltx25-distilled-i2v',
    'WAN2GP_SERVER_EAGER_INIT': '1',
    'WAN2GP_SERVER_API_KEY': secrets.token_urlsafe(20),
}
print('Profile: 1 (entire active model in VRAM)')
print('Default checkpoint: LTX-2.5 Distilled 22B')
print('API key:', SERVER_ENV['WAN2GP_SERVER_API_KEY'])

## 7. Start Frameflow and automatically download LTX-2.5

This cell starts FastAPI, waits for it to become healthy, then submits an LTX-2.5 warmup. The first run downloads the checkpoint and supporting model files; later runs can reuse Colab's cache while the runtime remains alive. The warmup finishes with LTX-2.5 loaded in VRAM.

In [ ]:
import os, subprocess, sys, time
import requests

BASE_URL = f'http://127.0.0.1:{PORT}'
LOG_PATH = Path('/content/frameflow_server.log')

if 'server_proc' in globals() and server_proc.poll() is None:
    print(f'Reusing running server PID {server_proc.pid}.')
else:
    server_env = os.environ.copy()
    server_env.update(SERVER_ENV)
    server_log = open(LOG_PATH, 'a')
    server_proc = subprocess.Popen(
        [sys.executable, '-m', 'wan2gp_server'],
        cwd=str(WAN2GP_ROOT), env=server_env,
        stdout=server_log, stderr=subprocess.STDOUT,
    )
    print(f'Server PID {server_proc.pid} · log {LOG_PATH}')

for _ in range(90):
    if server_proc.poll() is not None:
        print(LOG_PATH.read_text()[-5000:])
        raise RuntimeError('Frameflow server exited during startup.')
    try:
        health = requests.get(f'{BASE_URL}/health', timeout=2)
        if health.ok:
            print('Server online:', health.json())
            break
    except requests.RequestException:
        pass
    time.sleep(1)
else:
    raise RuntimeError(f'Server did not start. Inspect {LOG_PATH}.')

if str(WAN2GP_ROOT) not in sys.path:
    sys.path.insert(0, str(WAN2GP_ROOT))
from wan2gp_server.client import Wan2GPServerClient
client = Wan2GPServerClient(BASE_URL, api_key=SERVER_ENV['WAN2GP_SERVER_API_KEY'])

print('\nDownloading and warming LTX-2.5 Distilled. This can take several minutes…')
warmups = client.preload(['ltx25-distilled'], wait=True, timeout=4 * 3600, poll=5)
if not warmups or warmups[-1]['status'] != 'succeeded':
    raise RuntimeError(f'LTX-2.5 warmup did not succeed: {warmups}')
print('✓ LTX-2.5 Distilled is downloaded, warm, and ready in VRAM.')

## 8. Expose the Studio and API

The URL fragment carries the API key to Frameflow without sending it to Cloudflare as part of the HTTP request. The UI stores the key in this browser and removes the fragment from the address bar. Keep the Colab runtime alive while using the link.

In [ ]:
import os, queue, re, subprocess, threading
from IPython.display import HTML, display
from urllib.parse import quote

CLOUDFLARED = Path('/content/cloudflared')
if not CLOUDFLARED.exists():
    subprocess.run([
        'wget', '-q', '-O', str(CLOUDFLARED),
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'
    ], check=True)
    subprocess.run(['chmod', '+x', str(CLOUDFLARED)], check=True)

if 'tunnel_proc' in globals() and tunnel_proc.poll() is None:
    tunnel_proc.terminate()

tunnel_proc = subprocess.Popen(
    [str(CLOUDFLARED), 'tunnel', '--url', BASE_URL, '--protocol', 'http2', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
url_queue = queue.Queue()
def drain_tunnel_output():
    for line in iter(tunnel_proc.stdout.readline, ''):
        match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
        if match and url_queue.empty():
            url_queue.put(match.group(0))
threading.Thread(target=drain_tunnel_output, daemon=True).start()

try:
    public_url = url_queue.get(timeout=90)
except queue.Empty as exc:
    tunnel_proc.terminate()
    raise RuntimeError('Cloudflare tunnel did not return a URL within 90 seconds.') from exc

studio_url = f"{public_url}/#key={quote(SERVER_ENV['WAN2GP_SERVER_API_KEY'])}"
display(HTML(f'''
<div style="padding:20px 24px;border-radius:16px;background:#101d31;color:white;font-family:system-ui">
  <div style="font-size:12px;letter-spacing:.12em;color:#f26b51;font-weight:800">FRAMEFLOW IS READY</div>
  <a href="{studio_url}" target="_blank" style="display:inline-block;margin:12px 0 8px;padding:12px 18px;border-radius:10px;background:#f26b51;color:white;text-decoration:none;font-weight:700">Open Video Studio ↗</a>
  <div style="opacity:.65;font-size:12px">API docs: <a href="{public_url}/docs" target="_blank" style="color:#a9c6ef">{public_url}/docs</a></div>
</div>
'''))
print('Studio:', studio_url)
print('API key:', SERVER_ENV['WAN2GP_SERVER_API_KEY'])

## 9. Operations and troubleshooting

- Server log: `/content/frameflow_server.log`
- Generated media: `/content/frameflow_outputs`
- Studio/API health: `http://127.0.0.1:8000/health`
- Stop services with the optional cell below.
- If profile 1 runs out of VRAM, this GPU is not large enough for the requested high-VRAM configuration. Change `WAN2GP_CLI_ARGS` to `--profile 3 --attention sdpa` or use the low-VRAM notebook.
- A Cloudflare quick-tunnel URL is temporary and changes whenever the tunnel cell is rerun.

In [ ]:
# Optional: uncomment to stop both background services.
# if 'tunnel_proc' in globals() and tunnel_proc.poll() is None: tunnel_proc.terminate()
# if 'server_proc' in globals() and server_proc.poll() is None: server_proc.terminate()
# print('Frameflow services stopped.')